# Dimensión de grupos (membresía entidad × agrupamiento)

Construye `data/interim/dimensiones/dim_grupos.parquet`: tabla larga de pares (entidad, agrupamiento). Una entidad puede pertenecer a varios agrupamientos simultáneamente (por ejemplo, Banco de Galicia pertenece a `AA000` sistema financiero, `AA100` bancos, `AA120` privados, `AA121` de capital nacional, etc.).

La información viene de `Tec_Cont/entint/{codigo}.txt` de cada entidad, que lista a qué agrupamientos pertenece esa entidad al corte del dump.

Columnas:
- `codigo_entidad`: código BCRA de la entidad.
- `nombre_entidad`: redundante, para debugging.
- `yyyymm`: período al que refiere la membresía.
- `codigo_grupo`: código del agrupamiento (`AAxxx`).

Decisión metodológica: la membresía se toma del dump más reciente. Para análisis donde importe la membresía histórica (muy raro), habría que apilar los `entint/` de dumps sucesivos.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

ENTINT_DIR = RAW / "bcra_ief/202601/Entfin/Tec_Cont/entint"
OUT = DIMENSIONES / "dim_grupos.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

## Lectura

El archivo `entint/{cod}.txt` puede duplicar cada par (entidad, grupo) si el BCRA exporta la misma membresía dos veces (lo hace por errores de export). Se deduplica al cargar.

In [ ]:
bloques = []
for f in sorted(ENTINT_DIR.glob("*.txt")):
    if f.name == "formato.txt":
        continue
    df = pd.read_csv(f, sep="\t", header=None,
                     names=["codigo_entidad", "nombre_entidad", "yyyymm", "codigo_grupo"],
                     encoding="ISO-8859-1", dtype=str)
    bloques.append(df)

membresias = pd.concat(bloques, ignore_index=True)
print(f"Filas leídas: {len(membresias):,}")
membresias = membresias.drop_duplicates(subset=["codigo_entidad", "codigo_grupo"]).reset_index(drop=True)
print(f"Filas tras deduplicar por (entidad, grupo): {len(membresias):,}")
print(f"Entidades distintas: {membresias['codigo_entidad'].nunique()}")
print(f"Grupos distintos: {membresias['codigo_grupo'].nunique()}")
membresias.head(10)

## Persistencia

In [ ]:
membresias.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")

## Validaciones

In [ ]:
assert membresias["codigo_grupo"].str.startswith("AA").all(), "Códigos de grupo no empiezan con AA"
assert (membresias["codigo_entidad"].str.len() == 5).all(), "Códigos de entidad deben ser de 5 dígitos"

n_grupos_por_entidad = membresias.groupby("codigo_entidad").size()
print(f"Validaciones OK")
print(f"  Grupos promedio por entidad: {n_grupos_por_entidad.mean():.1f}")
print(f"  Mínimo: {n_grupos_por_entidad.min()}  Máximo: {n_grupos_por_entidad.max()}")

## Muestra con DuckDB

In [ ]:
duckdb.sql(f"""
    select codigo_entidad, nombre_entidad, codigo_grupo
    from '{OUT}'
    where codigo_entidad = '00007'
    order by codigo_grupo
""").df()